In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import joblib
import warnings
warnings.filterwarnings('ignore')

In [9]:
# TAHAP 1: BUSINESS UNDERSTANDING
# Tujuan (Objective): Menemukan pola tersembunyi atau kelompok (cluster) dari performa operasional kapal.
# Manfaat: Hasil segmentasi ini dapat membantu manajer armada untuk mengidentifikasi kapal yang sangat efisien, kapal yang boros biaya operasional, atau kapal dengan performa rata-rata, sehingga keputusan rute dan maintenance bisa lebih optimal.

In [11]:
# TAHAP 2: DATA UNDERSTANDING
# Memuat dataset
df = pd.read_csv("Ship_Performance_Dataset.csv")

# Menampilkan informasi dasar
print("Dimensi dataset:", df.shape)
print("\nInformasi Kolom:")
df.info()

Dimensi dataset: (2736, 18)

Informasi Kolom:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2736 entries, 0 to 2735
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Date                     2736 non-null   object 
 1   Ship_Type                2600 non-null   object 
 2   Route_Type               2600 non-null   object 
 3   Engine_Type              2600 non-null   object 
 4   Maintenance_Status       2600 non-null   object 
 5   Speed_Over_Ground_knots  2736 non-null   float64
 6   Engine_Power_kW          2736 non-null   float64
 7   Distance_Traveled_nm     2736 non-null   float64
 8   Draft_meters             2736 non-null   float64
 9   Weather_Condition        2600 non-null   object 
 10  Cargo_Weight_tons        2736 non-null   float64
 11  Operational_Cost_USD     2736 non-null   float64
 12  Revenue_per_Voyage_USD   2736 non-null   float64
 13  Turnaround_Time_hours    2736 no

In [12]:
# Menampilkan 5 baris pertama
display(df.head())

# Mengecek nilai yang kosong (missing values)
print("\nMissing Values:\n", df.isnull().sum())

,Date,Ship_Type,Route_Type,Engine_Type,Maintenance_Status,Speed_Over_Ground_knots,Engine_Power_kW,Distance_Traveled_nm,Draft_meters,Weather_Condition,Cargo_Weight_tons,Operational_Cost_USD,Revenue_per_Voyage_USD,Turnaround_Time_hours,Efficiency_nm_per_kWh,Seasonal_Impact_Score,Weekly_Voyage_Count,Average_Load_Percentage
0,2023-06-04,Container Ship,NaN,Heavy Fuel Oil (HFO),Critical,12.597558,2062.983982,1030.943616,14.132284,Moderate,1959.017882,483832.354540,292183.273104,25.867077,1.455179,1.415653,1,93.769249
1,2023-06-11,Fish Carrier,Short-haul,Steam Turbine,Good,10.387580,1796.057415,1060.486382,14.653083,Rough,162.394712,483388.000509,883765.787360,63.248196,0.290361,0.885648,6,93.895372
2,2023-06-18,Container Ship,Long-haul,Diesel,Fair,20.749747,1648.556685,658.874144,7.199261,Moderate,178.040917,448543.404044,394018.746904,49.418150,0.499595,1.405813,9,96.218244
3,2023-06-25,Bulk Carrier,Transoceanic,Steam Turbine,Fair,21.055102,915.261795,1126.822519,11.789063,Moderate,1737.385346,261349.605449,87551.375175,22.409110,0.702906,1.370704,1,66.193698
4,2023-07-02,Fish Carrier,Transoceanic,Diesel,Fair,13.742777,1089.721803,1445.281159,9.727833,Moderate,260.595103,287718.375160,676121.459632,64.158231,1.331343,0.583383,8,80.008581



Missing Values:
 Date                         0
Ship_Type                  136
Route_Type                 136
Engine_Type                136
Maintenance_Status         136
Speed_Over_Ground_knots      0
Engine_Power_kW              0
Distance_Traveled_nm         0
Draft_meters                 0
Weather_Condition          136
Cargo_Weight_tons            0
Operational_Cost_USD         0
Revenue_per_Voyage_USD       0
Turnaround_Time_hours        0
Efficiency_nm_per_kWh        0
Seasonal_Impact_Score        0
Weekly_Voyage_Count          0
Average_Load_Percentage      0
dtype: int64


In [13]:
# TAHAP 3: DATA PREPARATION
# Memilih minimal 3 fitur numerik untuk clustering sesuai ketentuan
features = ['Speed_Over_Ground_knots', 'Engine_Power_kW', 'Operational_Cost_USD', 'Revenue_per_Voyage_USD']
X = df[features].copy()

# Mengisi missing values (jika ada) dengan nilai median
X.fillna(X.median(), inplace=True)

# Standarisasi data agar skala tiap fitur seragam
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Menampilkan data yang sudah distandarisasi
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
display(X_scaled_df.head())

,Speed_Over_Ground_knots,Engine_Power_kW,Operational_Cost_USD,Revenue_per_Voyage_USD
0,-1.161003,0.425980,1.623465,-0.845173
1,-1.673617,0.053631,1.620311,1.336484
2,0.729934,-0.152125,1.372949,-0.469621
3,0.800762,-1.175035,0.044058,-1.599821
4,-0.895364,-0.931672,0.231250,0.570727
